**Contexte**

Le but de ce cas pratique est de vous familiariser avec la réalisation d'une analyse A/B testing en utilisant Python.

Spotify souhaite tester l'efficacité de sa nouvelle fonctionnalité "Smart Playlist Shuffle" qui propose une lecture aléatoire optimisée des playlists.

Votre mission est d'analyser les données pour déterminer si cette fonctionnalité améliore l'expérience utilisateur.

Explications sur les données :
- `users.csv` : contient les informations sur les utilisateurs (âge, pays, type d'abonnement).
- `experiments.csv` : décrit l'appartenance des utilisateurs aux groupes (groupe contrôle ou expérimental).
- `sessions.csv` : répertorie les sessions d'écoute des utilisateurs (durée des sessions, taux de skip des morceaux).

##Étape 1 : Chargement des jeux de données

Chargez les fichiers `users.csv`, `experiments.csv` et `sessions.csv` en DataFrame pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind


users = pd.read_csv("users.csv")  # Contient les informations des utilisateurs.
experiments = pd.read_csv("experiments.csv")  # Contient l'attribution des groupes.
sessions = pd.read_csv("sessions.csv")  # Contient les informations sur les sessions d'écoute.

In [ ]:
print(users.head())
print(experiments.head())
print(sessions.head())


##Étape 2 : Fusion des jeux de données

Fusionnez les tables `sessions`, `experiments` et `users` pour obtenir un DataFrame consolidé contenant les informations nécessaires pour l'analyse.

In [ ]:
df = pd.merge(sessions, experiments, on='user_id')
df = pd.merge(df, users, on='user_id')
df

##Étape 3 : Exploration des données

Affichez les premières lignes du DataFrame et les informations générales (types de colonnes, valeurs manquantes, etc.).

In [ ]:
df.info()

In [ ]:
df.head()

## **Étape 4 : Vérification des doublons**

Identifiez les doublons (mêmes `user_id`, `timestamp_start` et `timestamp_end`).

Combien y a-t-il de doublons ?

In [ ]:
df[df.duplicated]

Supprimez les doublons si nécessaire.

In [ ]:
df.drop_duplicates(keep = 'first')

##Étape 5 : Analyse des valeurs aberrantes

In [ ]:
df.columns

Affichez un boxplot pour observer la distribution des durées des sessions.

In [ ]:
box = sns.boxplot(df['duration_sec'])

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
plt.boxplot(df['duration_sec'])
plt.ylabel("Durée des sessions")
plt.title("Mon jolie Boxplot")
plt.show()

Calculez les bornes inférieures et supérieures pour détecter les valeurs aberrantes.

D'après vous, pourquoi voudrions nous trouver et retirer les valeurs aberrantes ?


In [ ]:
df['duration_sec'].describe()

borne inferieure = Q1 - 1.5 * IQR

In [ ]:
Q3 = df['duration_sec'].quantile(0.75)
Q1 = df['duration_sec'].quantile(0.25)
IQR = Q3 - Q1
borne_inferieure = Q1 - 1.5 * IQR
print(borne_inferieure)

borne superieure = Q3 + 1.5 * IQR

In [ ]:
borne_superieure = Q3 + 1.5 * IQR
print(borne_superieure)

Filtrez les sessions pour retirer les valeurs aberrantes.

In [ ]:
test = df[df['duration_sec'] < borne_superieure]
test

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
plt.boxplot(test["duration_sec"])
plt.ylabel("Durée des sessions")
plt.title("Mon jolie Boxplot amélioré")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
sns.boxplot (hue = 'group', y="duration_sec", data=test)
plt.ylabel("Durée des sessions")
plt.title("Mon jolie Boxplot amélioré")
plt.show()

In [ ]:
df = test

##Étape 6 : Visualisation de la distribution des sessions par groupe

Affichez un histogramme comparant la distribution des durées d'écoute pour les groupes contrôle et expérimental.

In [ ]:
df["group"]

In [ ]:
plt.hist(df[df['group'] == 'control']['duration_sec'], alpha=0.6, label='Control')
plt.hist(df[df['group'] == 'treatment']['duration_sec'], alpha=0.4, label='Experiment')
plt.legend()
plt.xlabel('Durée des sessions')
plt.ylabel('Fréquence')
plt.title('Distribution des durées des sessions par groupe')
plt.show()

##Étape 7 : Visualisation du taux de skip par groupe

Affichez un boxplot du taux de skip par groupe.

In [ ]:
skip_control = df[df['group'] == 'control']['skipped_rate']
skip_treatment = df[df['group'] == 'treatment']['skipped_rate']

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
plt.boxplot(skip_control)
plt.ylabel("Taux de skip")
plt.title("Box control")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
plt.boxplot(skip_treatment)
plt.ylabel("Taux de skip")
plt.title("Box treatment")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
sns.boxplot (hue = 'group', y='skipped_rate', data=df)
plt.ylabel("Taux de skip")
plt.title("Box treatment")
plt.ylim(-0.1,0.5)
plt.axhline(0, color='red', linestyle='--')
plt.show()

##Étape 8 : Calcul des métriques clés par groupe

Calculez la durée moyenne des sessions, le nombre moyen de morceaux joués et le taux moyen de morceaux "skippés" pour chaque groupe.

In [ ]:
df.columns

In [ ]:
df.groupby('group').agg({'duration_sec': 'mean', 'skipped_rate': 'mean', 'tracks_played': 'mean'})

##Étape 9 : Test statistique A/B

Comparez statistiquement la durée moyenne des sessions entre nos 2 échantillons.

Automatisez une phrase réponse indiquant si la différence est statistiquement significative ou non. Si elle l'est, précisez quel groupe a la durée moyenne des sessions la plus élevée.

In [ ]:
df_control = df[df['group'] == 'control']
df_treatment = df[df['group'] == 'treatment']

In [ ]:
control = df[df['group'] == 'control']['duration_sec']
treatment = df[df['group'] == 'treatment']['duration_sec']
t_stat,p_value= ttest_ind(control, treatment, equal_var=False)
print(t_stat)
print(p_value)

Même consigne pour le taux de skip.

In [ ]:
control_2 = df[df['group'] == 'control']['skipped_rate']
treatment_2 = df[df['group'] == 'treatment']['skipped_rate']
t_stat,p_value= ttest_ind(control_2, treatment_2, equal_var=False)
print(t_stat)
print(p_value)

##Étape 10 : Analyse par type d'abonnement

Rentrons maintenant dans le détail.

Effectuez un test t pour comparer la durée des sessions au sein de chaque type d'abonnement.

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(10, 5))  # Largeur : 10, Hauteur : 5
sns.boxplot (hue = 'subscription_type', y='duration_sec', data=df)
plt.ylabel("Durée d'écoute ")
plt.title("Box")
plt.show()

## Étape 11 : Analyse du type d'abonnement

Effectuez un test t pour comparer la durée des sessions en fonction du type d'abonnement (`subscription_type`).

In [ ]:
df.groupby('subscription_type').agg({'duration_sec': 'mean', 'skipped_rate': 'mean', 'tracks_played': 'mean'})

In [ ]:
df['subscription_type'].unique()

In [ ]:
free = df[df['subscription_type'] == 'free']['duration_sec']
prenium = df[df['subscription_type'] == 'premium']['duration_sec']
t_stat,p_value= ttest_ind(free, prenium, equal_var=False)
print(t_stat)
print(p_value)

In [ ]:
free_2 = df[df['subscription_type'] == 'free']['skipped_rate']
prenium_2 = df[df['subscription_type'] == 'premium']['skipped_rate']
t_stat,p_value= ttest_ind(free_2, prenium_2, equal_var=False)
print(t_stat)
print(p_value)

In [ ]:
free_3 = df[df['subscription_type'] == 'free']['tracks_played']
prenium_3 = df[df['subscription_type'] == 'premium']['tracks_played']
t_stat,p_value= ttest_ind(free_3, prenium_3, equal_var=False)
print(t_stat)
print(p_value)

##Étape 12 : Analyses complémentaires

Après ces premiers résultats d'A/B testing concluant, le Product Manager du projet Spotify Smart Playlist Shuffle vous demande de lui faire une présentation plus complète sur le sujet et vous recevez l'email suivant :

"En utilisant les données fournies, identifiez un ou plusieurs segments d’utilisateurs ou contextes d’écoute pour lesquels la fonctionnalité "Smart Playlist Shuffle" pourrait avoir un impact différencié. Formulez une hypothèse, réalisez une analyse et présentez moi vos conclusions en fin de journée en m'expliquant dans quels contextes cette fonctionnalité améliore l'expérience utilisateur (ou non)."

Pour cela, à vous de continuer à creuser le sujet en effectuant des analyses complémentaires. Vous pourrez par exemple:



*  Vérifier si la fonctionnalité "Smart Playlist Shuffle" a un impact différent en fonction du pays des utilisateurs, de leur tranche d'âge, ...
*  Faire une comparaison selon la longueur des sessions d'écoute
*  Regarder l'effet global sur le taux de retour
*  Observer si l’impact de la fonctionnalité évolue au fil du temps
* ...